# BCC noncollinear QE input from the last dataset step

This notebook follows the established BCC QE-input workflow and creates a noncollinear MD input with Maxwell–Boltzmann velocities.

It:

1. reads `dataset/bcc/magnetic-non_coll/simulation.npz`;
2. selects the last **complete** trajectory step by default (iteration 66 in the current archive);
3. copies that step's atomic positions and cell into a 54-atom BCC noncollinear QE input;
4. creates a zero-net magnetic SQS and one QE species label per spin direction;
5. samples Maxwell–Boltzmann velocities, removes center-of-mass drift, rescales them to exactly 4000 K, and inserts `ATOMIC_VELOCITIES { a.u }`;
6. verifies the positions, cell, labels, center-of-mass velocity, and kinetic temperature.

The archive also contains a final position-only record at iteration 67. It is excluded by default because QE stopped during its SCF before producing complete forces and local spins. Set `USE_LAST_COMPLETE_FRAME = False` below only if that raw final position record is specifically desired.

In [ ]:
from pathlib import Path
from pprint import pprint
import importlib
import json
import sys

import numpy as np

# Locate the workspace whether Jupyter starts in Fe/ or IronCoreMD/codes/.
CWD = Path.cwd().resolve()
WORKSPACE_ROOT = None
for candidate in (CWD, *CWD.parents):
    if (candidate / "IronCoreMD" / "codes" / "prepare_latest_bcc_qe_input.py").exists():
        WORKSPACE_ROOT = candidate
        break
if WORKSPACE_ROOT is None:
    raise FileNotFoundError("Could not locate the Fe workspace root.")

REPO_ROOT = WORKSPACE_ROOT / "IronCoreMD"
CODES_DIR = REPO_ROOT / "codes"
if str(CODES_DIR) in sys.path:
    sys.path.remove(str(CODES_DIR))
sys.path.insert(0, str(CODES_DIR))

import prepare_latest_bcc_qe_input as bcc_qe
import generate_qe_maxwell_velocities as qe_velocity
bcc_qe = importlib.reload(bcc_qe)
qe_velocity = importlib.reload(qe_velocity)

DATASET_DIR = WORKSPACE_ROOT / "dataset" / "bcc" / "magnetic-non_coll"
NPZ_PATH = DATASET_DIR / "simulation.npz"
OUTPUT_DIR = WORKSPACE_ROOT / "prepared_qe_inputs" / "bcc_noncollinear_last_frame_4000K"

USE_LAST_COMPLETE_FRAME = True
TARGET_TEMPERATURE_K = 4000.0
VELOCITY_SEED = 400054
SPIN_SEED = 20260730
STARTING_MAGNETIZATION = 0.35

PSEUDO_DIR = WORKSPACE_ROOT
PSEUDO_FILE = "Fe.pbe-spn-kjpaw_psl.1.0.0.UPF"
DT_AU = 20.670
NSTEP = 400
ECUTWFC = 71.0
ECUTRHO = 496.0
DEGAUSS = 0.02
K_GRID = (1, 1, 1, 0, 0, 0)
MIXING_BETA = 0.01
CONSTRAINED_MAGNETIZATION = True
LAMBDA_VALUE = 0.2

print(f"Dataset: {NPZ_PATH}")
print(f"Output:  {OUTPUT_DIR}")

In [ ]:
# Select the last complete synchronized step, or the literal final position record.
with np.load(NPZ_PATH, allow_pickle=False) as data:
    nframes, natoms, _ = data["positions"].shape
    iterations = np.asarray(data["iteration"], dtype=int)
    times_ps = np.asarray(data["time_ps"], dtype=float)
    temperatures_k = np.asarray(data["temperature_K"], dtype=float)

    valid = np.asarray(data.get("position_frame_valid", np.ones(nframes, dtype=bool)), dtype=bool)
    if "frame_valid" in data.files:
        valid &= np.asarray(data["frame_valid"], dtype=bool)
    if "local_magnetization_frame_valid" in data.files:
        valid &= np.asarray(data["local_magnetization_frame_valid"], dtype=bool)

    if USE_LAST_COMPLETE_FRAME:
        valid_indices = np.flatnonzero(valid)
        if valid_indices.size == 0:
            raise ValueError("The dataset contains no complete synchronized frames.")
        FRAME_INDEX = int(valid_indices[-1])
    else:
        FRAME_INDEX = nframes - 1

print(f"Frames in archive:       {nframes}")
print(f"Atoms per frame:         {natoms}")
print(f"Selected frame index:    {FRAME_INDEX}")
print(f"Selected QE iteration:   {iterations[FRAME_INDEX]}")
print(f"Selected time:           {times_ps[FRAME_INDEX]:.6f} ps")
print(f"Instantaneous dataset T: {temperatures_k[FRAME_INDEX]:.3f} K")
print(f"Velocity target T:       {TARGET_TEMPERATURE_K:.1f} K")
print(f"Selected frame complete: {bool(valid[FRAME_INDEX])}")

if FRAME_INDEX != nframes - 1:
    print(
        f"Skipped raw final record: frame {nframes - 1}, iteration {iterations[-1]}, "
        f"complete={bool(valid[-1])}"
    )

In [ ]:
# Build the noncollinear QE base file, compute velocities, and patch the velocity card.
result = bcc_qe.prepare_bcc_qe_input(
    dataset_dir=DATASET_DIR,
    npz_path=NPZ_PATH,
    output_dir=OUTPUT_DIR,
    frame_index=FRAME_INDEX,
    temperature_k=TARGET_TEMPERATURE_K,
    velocity_seed=VELOCITY_SEED,
    spin_seed=SPIN_SEED,
    magnetic_mode=bcc_qe.MAGNETIC_MODE_NONCOLLINEAR_RANDOM,
    spin_mode=bcc_qe.SPIN_MODE_MAGNETIC_SQS,
    m_abs=STARTING_MAGNETIZATION,
    pseudo_dir=str(PSEUDO_DIR),
    pseudo_file=PSEUDO_FILE,
    dt_au=DT_AU,
    nstep=NSTEP,
    ecutwfc=ECUTWFC,
    ecutrho=ECUTRHO,
    degauss=DEGAUSS,
    k_grid=K_GRID,
    constrained_magnetization=CONSTRAINED_MAGNETIZATION,
    lambda_value=LAMBDA_VALUE,
    mixing_beta=MIXING_BETA,
    remove_com_drift=True,
    rescale_exact=True,
    nosym=True,
)

pprint(result.as_dict())

In [ ]:
# Verify that the QE positions and cell are exactly the selected dataset frame.
def read_qe_position_card(path: Path, natoms: int) -> tuple[list[str], np.ndarray]:
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip().upper().startswith("ATOMIC_POSITIONS"))
    labels = []
    positions = []
    for line in lines[start + 1 : start + 1 + natoms]:
        fields = line.split()
        labels.append(fields[0])
        positions.append([float(value) for value in fields[1:4]])
    return labels, np.asarray(positions)

def read_qe_cell_card(path: Path) -> np.ndarray:
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip().upper().startswith("CELL_PARAMETERS"))
    return np.asarray([[float(value) for value in lines[start + offset].split()[:3]] for offset in (1, 2, 3)])

with np.load(NPZ_PATH, allow_pickle=False) as data:
    fallback_cell = bcc_qe.fixed_cell_angstrom(data)
    expected_cell = bcc_qe.frame_cell_angstrom(data, FRAME_INDEX, fallback_cell)
    expected_positions = bcc_qe.wrap_fractional(
        bcc_qe.frame_positions_fractional(data, FRAME_INDEX, expected_cell)
    )

position_labels, generated_positions = read_qe_position_card(Path(result.qe_input_final), natoms)
generated_cell = read_qe_cell_card(Path(result.qe_input_final))

np.testing.assert_allclose(generated_positions, expected_positions, atol=5.1e-11, rtol=0.0)
np.testing.assert_allclose(generated_cell, expected_cell, atol=5.1e-11, rtol=0.0)
print("PASS: generated ATOMIC_POSITIONS match the selected dataset frame")
print("PASS: generated CELL_PARAMETERS match the selected dataset frame")
print(f"Position range (fractional): {generated_positions.min():.8f} -> {generated_positions.max():.8f}")

In [ ]:
# Verify the computed QE velocities after their text-format round trip.
velocity_labels, velocities_au = qe_velocity.parse_atomic_velocities(
    Path(result.qe_input_final).read_text().splitlines()
)
species_labels, masses_amu = qe_velocity.read_qe_species_sequence(Path(result.qe_input_final))
measured_temperature = qe_velocity.temperature_from_velocities_au(
    velocities_au, masses_amu, remove_com=True
)
mass_weighted_velocity_sum = np.sum(masses_amu[:, None] * velocities_au, axis=0)

assert species_labels == position_labels == velocity_labels
np.testing.assert_allclose(mass_weighted_velocity_sum, 0.0, atol=1.0e-12, rtol=0.0)
np.testing.assert_allclose(measured_temperature, TARGET_TEMPERATURE_K, atol=1.0e-8, rtol=0.0)

print("PASS: ATOMIC_POSITIONS and ATOMIC_VELOCITIES labels match line-by-line")
print(f"PASS: mass-weighted COM velocity sum = {mass_weighted_velocity_sum}")
print(f"PASS: velocity temperature = {measured_temperature:.9f} K")

In [ ]:
# Save a compact provenance record and preview the generated cards.
summary = result.as_dict()
summary.update(
    {
        "selected_iteration": int(iterations[FRAME_INDEX]),
        "selected_time_ps": float(times_ps[FRAME_INDEX]),
        "selected_frame_complete": bool(valid[FRAME_INDEX]),
        "dataset_instantaneous_temperature_k": float(temperatures_k[FRAME_INDEX]),
        "use_last_complete_frame": USE_LAST_COMPLETE_FRAME,
    }
)
summary_path = OUTPUT_DIR / "generation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")

final_input = Path(result.qe_input_final)
velocity_block = Path(result.velocity_block)
print(f"Final QE input: {final_input}")
print(f"Velocity card:  {velocity_block}")
print(f"Summary:        {summary_path}")

print("\nATOMIC_POSITIONS preview:")
for label, position in list(zip(position_labels, generated_positions))[:5]:
    print(f"{label:4s} {position[0]: .10f} {position[1]: .10f} {position[2]: .10f}")

print("\nATOMIC_VELOCITIES preview:")
print("\n".join(velocity_block.read_text().splitlines()[:6]))